## Example 1: BaseInstrumentor Framework and Custom Instrumentor Creation

In [ ]:
from opentelemetry.instrumentation.instrumentor import BaseInstrumentor
from opentelemetry.instrumentation.version import __version__
from opentelemetry import trace
from typing import Collection
import sys

try:
    # Example 1: Custom instrumentor creation using BaseInstrumentor framework
    print("=== OpenTelemetry Instrumentation Framework - BaseInstrumentor ===")
    
    # Create a custom instrumentor for a hypothetical library
    class DemoLibraryInstrumentor(BaseInstrumentor):
        """Custom instrumentor for demonstration purposes"""
        
        def instrumentation_dependencies(self) -> Collection[str]:
            """Return dependencies required for instrumentation"""
            return ["opentelemetry-api >= 1.1.0"]
        
        def _instrument(self, **kwargs):
            """Implement the actual instrumentation logic"""
            tracer_provider = kwargs.get("tracer_provider")
            tracer = trace.get_tracer(__name__, __version__, tracer_provider)
            
            # Simulate instrumenting a library function
            self._original_function = getattr(sys.modules[__name__], 'demo_function', None)
            
            def instrumented_function(*args, **kwargs):
                with tracer.start_as_current_span("demo_library_operation") as span:
                    span.set_attribute("demo.operation", "custom_instrumentation")
                    return f"Instrumented result: {args}, {kwargs}"
            
            # Store reference for uninstrumentation
            self._instrumented_function = instrumented_function
            print(f"✅ Custom instrumentor applied to demo_function")
            
        def _uninstrument(self, **kwargs):
            """Remove instrumentation"""
            if hasattr(self, '_original_function') and self._original_function:
                # Restore original function (simplified demo)
                print(f"✅ Custom instrumentor removed")
            else:
                print(f"✅ Uninstrumentation completed (no original function to restore)")
    
    # Demonstrate instrumentor lifecycle
    demo_instrumentor = DemoLibraryInstrumentor()
    
    print(f"Instrumentation framework version: {__version__}")
    print(f"Custom instrumentor class: {demo_instrumentor.__class__.__name__}")
    print(f"Dependencies: {demo_instrumentor.instrumentation_dependencies()}")
    
    # Check instrumentation status
    is_instrumented = demo_instrumentor.is_instrumented_by_opentelemetry
    print(f"Currently instrumented: {is_instrumented}")
    
    # Apply instrumentation
    demo_instrumentor.instrument()
    print(f"Instrumentation applied successfully")
    
    # Check status again
    is_instrumented_after = demo_instrumentor.is_instrumented_by_opentelemetry
    print(f"Instrumented after application: {is_instrumented_after}")
    
    # Test the instrumented functionality
    if hasattr(demo_instrumentor, '_instrumented_function'):
        result = demo_instrumentor._instrumented_function("test", param="value")
        print(f"Instrumented function result: {result}")
    
    # Remove instrumentation
    demo_instrumentor.uninstrument()
    print(f"✅ BaseInstrumentor framework demonstration completed")
    
except Exception as e:
    raise AssertionError(
        f"opentelemetry_instrumentation example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Instrumentation Utilities and Function Wrapping

In [ ]:
from opentelemetry.instrumentation.utils import unwrap, http_status_to_status_code
from opentelemetry.trace import Status, StatusCode
import functools

try:
    # Example 2: Instrumentation utilities for function wrapping and HTTP handling
    print("=== OpenTelemetry Instrumentation - Utilities and Wrapping ===")
    
    # Create a sample function to instrument
    def sample_business_function(x, y):
        """Sample business logic function"""
        return x * y + 42
    
    # Create a wrapper function with instrumentation
    def create_instrumented_wrapper(func):
        """Create an instrumented wrapper for any function"""
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            print(f"🔧 Instrumented call to {func.__name__} with args={args}, kwargs={kwargs}")
            try:
                result = func(*args, **kwargs)
                print(f"✅ Function completed successfully: {result}")
                return result
            except Exception as e:
                print(f"❌ Function failed: {e}")
                raise
        return wrapper
    
    # Apply instrumentation wrapper
    original_function = sample_business_function
    instrumented_function = create_instrumented_wrapper(sample_business_function)
    
    print(f"Original function: {original_function.__name__}")
    print(f"Instrumented function: {instrumented_function.__name__}")
    
    # Test the instrumented function
    result1 = instrumented_function(5, 3)
    print(f"Instrumentation result: {result1}")
    
    # Demonstrate unwrap utility
    try:
        # Simulate unwrapping (though our wrapper wasn't applied with setattr)
        print(f"✅ Unwrap utility available for removing instrumentation")
        print(f"Unwrap function: {unwrap}")
        
        # Test HTTP status code utility
        http_statuses = [200, 404, 500, 302, 201]
        print(f"\n🌐 HTTP Status Code to OpenTelemetry Status mapping:")
        for status in http_statuses:
            otel_status = http_status_to_status_code(status)
            print(f"  HTTP {status} → OpenTelemetry {otel_status}")
            
        # Demonstrate Status objects
        success_status = Status(StatusCode.OK, "Operation completed successfully")
        error_status = Status(StatusCode.ERROR, "Operation failed")
        
        print(f"\n📊 OpenTelemetry Status objects:")
        print(f"Success status: {success_status.status_code} - {success_status.description}")
        print(f"Error status: {error_status.status_code} - {error_status.description}")
        
        # Practical instrumentation pattern
        def http_request_simulator(status_code):
            """Simulate HTTP request with status mapping"""
            otel_status_code = http_status_to_status_code(status_code)
            if otel_status_code == StatusCode.ERROR:
                return Status(StatusCode.ERROR, f"HTTP {status_code} error")
            else:
                return Status(StatusCode.OK, f"HTTP {status_code} success")
        
        # Test with various HTTP codes
        test_codes = [200, 404, 500]
        print(f"\n🚀 HTTP Request Simulation with Status Mapping:")
        for code in test_codes:
            status = http_request_simulator(code)
            print(f"  HTTP {code}: {status.status_code} - {status.description}")
            
    except Exception as e:
        print(f"⚠️ Utility function testing: {e}")
        raise
    
    print(f"✅ Instrumentation utilities demonstration completed")
    
except Exception as e:
    raise AssertionError(
        f"opentelemetry_instrumentation example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Semantic Conventions and Dependency Management

In [ ]:
from opentelemetry.instrumentation._semconv import CLIENT_ADDRESS, SERVER_ADDRESS, SERVER_PORT
from opentelemetry.instrumentation.dependencies import get_dist_dependency_conflicts
import importlib.metadata

try:
    # Example 3: Semantic conventions and dependency conflict detection
    print("=== OpenTelemetry Instrumentation - Semantic Conventions & Dependencies ===")
    
    # Demonstrate semantic conventions
    print(f"🏷️ Semantic Convention Constants:")
    print(f"Client address: {CLIENT_ADDRESS}")
    print(f"Server address: {SERVER_ADDRESS}")
    print(f"Server port: {SERVER_PORT}")
    
    # Demonstrate dependency checking
    print(f"\n📦 Dependency Management:")
    
    # Test dependency conflict detection: the function takes an importlib.metadata Distribution (not a
    # name) and returns a single DependencyConflict, or None when the dist's requirements are satisfied
    conflicts = get_dist_dependency_conflicts(importlib.metadata.distribution("opentelemetry-instrumentation"))
    if conflicts:
        print(f"⚠️ Dependency conflict found: {conflicts}")
    else:
        print(f"✅ No dependency conflicts detected")
    assert conflicts is None, conflicts
    
    # Demonstrate semantic convention usage
    def create_span_attributes(client_ip, server_host, server_port):
        """Create span attributes using semantic conventions"""
        return {
            CLIENT_ADDRESS: client_ip,
            SERVER_ADDRESS: server_host,
            SERVER_PORT: server_port
        }
    
    # Test semantic convention usage
    attrs = create_span_attributes("192.168.1.100", "api.example.com", 443)
    print(f"\n🏷️ Span attributes with semantic conventions:")
    for key, value in attrs.items():
        print(f"  {key}: {value}")
    
    print(f"✅ Semantic conventions demonstrated")
    
except Exception as e:
    raise AssertionError(
        f"opentelemetry_instrumentation example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Distro configuration and instrumentation suppression

In [ ]:
from opentelemetry.instrumentation.distro import BaseDistro
from opentelemetry.instrumentation.utils import is_instrumentation_enabled, suppress_instrumentation

try:
    # Example 4: Distro configuration and instrumentation suppression
    print("=== OpenTelemetry Instrumentation - Distro and Suppression ===")

    # A distro is what auto-instrumentation loads first; configure() hands its keyword
    # arguments to the subclass's _configure()
    class DemoDistro(BaseDistro):
        def _configure(self, **kwargs):
            self.configured_with = dict(kwargs)

    distro = DemoDistro()
    distro.configure(service_name="demo-instrumentation-service", auto_instrument=True)
    print(f"Distro {type(distro).__name__} configured with: {distro.configured_with}")
    assert distro.configured_with == {"service_name": "demo-instrumentation-service", "auto_instrument": True}
    assert callable(distro.load_instrumentor)

    # suppress_instrumentation() switches instrumentation off for the current context only
    assert is_instrumentation_enabled()
    with suppress_instrumentation():
        assert not is_instrumentation_enabled()
        print("Inside suppress_instrumentation(): instrumentation disabled")
    assert is_instrumentation_enabled()
    print("After the block: instrumentation enabled again")

    print(f"✅ Distro configuration and instrumentation suppression checked")

except Exception as e:
    raise AssertionError(
        f"opentelemetry_instrumentation example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Managing several instrumentors from a registry

In [ ]:
import json
import threading
from typing import Any, Dict

try:
    # Example 5: Managing several instrumentors from a registry
    # Reuses DemoLibraryInstrumentor from Example 1.
    print("=== OpenTelemetry Instrumentation - Instrumentor Registry ===")
    
    # Registry that instruments and uninstruments a set of instrumentors together
    class InstrumentorRegistry:
        """Registry for managing multiple instrumentors"""
        
        def __init__(self):
            self._instrumentors: Dict[str, BaseInstrumentor] = {}
            self._lock = threading.Lock()
            
        def register(self, name: str, instrumentor: BaseInstrumentor):
            """Register an instrumentor"""
            with self._lock:
                self._instrumentors[name] = instrumentor
                print(f"📝 Registered instrumentor: {name}")
                
        def instrument_all(self, **kwargs):
            """Instrument all registered instrumentors (a failure propagates: nothing here is optional)"""
            with self._lock:
                for name, instrumentor in self._instrumentors.items():
                    if not instrumentor.is_instrumented_by_opentelemetry:
                        instrumentor.instrument(**kwargs)
                        print(f"✅ Instrumented: {name}")

        def uninstrument_all(self):
            """Remove all instrumentation"""
            with self._lock:
                for name, instrumentor in self._instrumentors.items():
                    if instrumentor.is_instrumented_by_opentelemetry:
                        instrumentor.uninstrument()
                        print(f"🔄 Uninstrumented: {name}")
                        
        def get_status(self) -> Dict[str, Any]:
            """Get instrumentation status for all registered instrumentors"""
            status = {}
            with self._lock:
                for name, instrumentor in self._instrumentors.items():
                    status[name] = {
                        "instrumented": instrumentor.is_instrumented_by_opentelemetry,
                        "dependencies": list(instrumentor.instrumentation_dependencies()),
                        "class": instrumentor.__class__.__name__
                    }
            return status
    
    # Create the registry
    registry = InstrumentorRegistry()
    
    # Register demo instrumentors
    demo_instrumentor1 = DemoLibraryInstrumentor()
    demo_instrumentor2 = DemoLibraryInstrumentor()
    
    registry.register("http_client", demo_instrumentor1)
    registry.register("database", demo_instrumentor2)
    
    # Get initial status
    initial_status = registry.get_status()
    print(f"📊 Initial instrumentation status:")
    print(json.dumps(initial_status, indent=2))
    # Example 1 ended with uninstrument(), so nothing is active yet
    assert not any(s["instrumented"] for s in initial_status.values()), initial_status
    
    # Apply instrumentation
    registry.instrument_all()
    
    # Get status after instrumentation
    instrumented_status = registry.get_status()
    print(f"\n📊 Status after instrumentation:")
    for name, status in instrumented_status.items():
        print(f"  {name}: {'✅ Active' if status['instrumented'] else '❌ Inactive'}")

    # BaseInstrumentor is a per-class singleton: both registry entries are the SAME object, so one
    # instrument() call activates both. instrument() LOGS a dependency conflict and returns None instead of
    # raising, so this status check is what catches an instrumentor that silently stayed inactive.
    assert demo_instrumentor1 is demo_instrumentor2
    assert all(s["instrumented"] for s in instrumented_status.values()), instrumented_status
    
    # Cleanup
    registry.uninstrument_all()
    assert not any(s["instrumented"] for s in registry.get_status().values()), registry.get_status()
    
    print(f"\n✅ Registry instrument/uninstrument cycle checked")
    
except Exception as e:
    raise AssertionError(
        f"opentelemetry_instrumentation example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e